# 2026.1.1代码练习

## hello ollama

In [ ]:
from itertools import chain

from langchain.chat_models import init_chat_model

In [ ]:
model = init_chat_model(
    model="ollama:deepseek-r1:7b",
    base_url="http://localhost:11434/",
    temperature=0.5,
    timeout=20,
    max_tries=1000,
)

In [ ]:
for chunk in model.stream("给我一首流行歌曲的歌词。"):
    print(chunk.content, end="", flush=True)

## hello deepseek

In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

In [ ]:
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=api_key,
    temperature=0.5,
)
for chunk in model.stream("给我一首完整古代诗词。"):
    print(chunk.content, end="", flush=True)

## semantic_search  语义搜索

1.读取pdf 按照页管理  Document  list[Document]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
pdf_path = '03_text.pdf'
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs))
print(type(docs[0]))
print(docs[0])

2.分割文本  文本段（chunk）  Document  list[Document]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
print(len(all_splits))
print(all_splits[10])

3.向量化： 文本段<=>向量   需要嵌入模型辅助

In [ ]:
from langchain_ollama import OllamaEmbeddings

In [ ]:
embedding = OllamaEmbeddings(
    model="qwen3-embedding:8b"
)
vector_0 = embedding.embed_query(all_splits[10].page_content)
print(len(vector_0))  # qwen3-embedding:8b  4096 与nomic-embed-text:latest  768
print(vector_0)

4（3）.向量库：把多个文本段/向量存入向量库(融合第三步)

In [ ]:
from langchain_chroma import Chroma

In [ ]:
vector_store = Chroma(
    collection_name='example_03_collection',
    embedding_function=embedding,  # 用到第三步建立的嵌入模型
    persist_directory='./chroma_lc_db_03'
)
ids = vector_store.add_documents(documents=all_splits)
print(len(ids))
print(ids)

注意：
1.运行会记录存储内容，不可过多运行，会累计；可以删除目录，重建向量库；
2.collection_name的作用：集合标识符、 数据隔离、恢复、连接现有集合。

## 相似度查询

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [ ]:
# 嵌入模型
embedding = OllamaEmbeddings(
    model="qwen3-embedding:8b"
)

# 向量库
vector_store = Chroma(
    collection_name='example_03_collection',
    embedding_function=embedding,
    persist_directory='./chroma_lc_db_03'
)

1.基础的相似度查询

In [ ]:
results = vector_store.similarity_search("安全事故的分类。")
for i, result in enumerate(results):
    print(f'{i}---{result}')

2.带分数的相似度查询

In [ ]:
results = vector_store.similarity_search_with_score("安全事故的分类。")
for result, score in results:
    print(f'{score}---{result}')
# 返回的是元组，用for循环解包；score数越小，相似度越高；参数k默认为返回前4条

3.用向量进行相似度查询(把输入转化为向量，再到向量数据库查询。与第一种等价，相当于拆分)

In [ ]:
vector = embedding.embed_query("安全事故的分类。")
results = vector_store.similarity_search_by_vector(vector)
for i, result in enumerate(results, 1):
    print(f'{i}-{result.page_content[:50]}')

LCEL: 大模型、提示词模板、tools、output 等串成链。Runnable

In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import chain

In [ ]:
# 用检索器查询，把相似度查询封装成检索器
@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)  # k是返回的数量，默认4

In [ ]:
results = retriever.invoke("安全事故的分类。")
for i, result in enumerate(results, 1):
    print(f'{i}-{result.page_content[:50]}')

## chromadb tool

In [ ]:
import chromadb

列出向量库的collections和记录

In [ ]:
def list_collection(db_path):
    client = chromadb.PersistentClient(db_path)
    collections = client.list_collections()
    print(f'chromadb:{db_path}--{len(collections)}个collections')
    for i, col in enumerate(collections, 1):
        print(f'{i}--{col.name}--有{col.count()}条')

In [ ]:
db_path = './chroma_lc_db_03'
list_collection(db_path)

删除操作

In [ ]:
def delete_collection(db_path, collection_name):
    try:
        client = chromadb.PersistentClient(db_path)
        client.delete_collection(collection_name)

    except Exception as e:
        print(e)

## chromadb score

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
# 嵌入模型
embedding = OllamaEmbeddings(
    model='qwen3-embedding:8b'
)

In [ ]:
# 评分方式
score_measures = [
    'default',
    'cosine',  # 用两个向量的夹角度量相似度
    'l2',  # 用两个向量的距离度量相似度
    'ip'  # 用两个向量的内积/点积度量相似度
]

创建向量库和4个collection

In [ ]:
persist_dir = '06_chroma_score_db'
vector_stores = []
for score_measure in score_measures:
    collection_metadata = {'hnsw:space': score_measure}
    if score_measure == 'default':
        collection_metadata = None

    collection_name = f'collection_{score_measure}'
    vector_stores.append(Chroma(
        collection_name=collection_name,
        embedding_function=embedding,
        persist_directory=persist_dir,
        collection_metadata=collection_metadata
    ))

In [ ]:
def indexing(docs):
    for vector_store in vector_stores:
        ids = vector_store.add_documents(documents=docs)
        print(f'\n集合：{vector_store._collection.name}')
        print(ids)

准备docs用于写入

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = '03_text.pdf'
loader = PyPDFLoader(pdf_path)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

In [ ]:
indexing(all_splits)

In [ ]:
def query_with_score(query):
    for i in range(len(score_measures)):
        results = vector_stores[i].similarity_search_with_score(query, k=1)
        print(f'\n搜索：{query}')
        for doc, score in results:
            print(doc.page_content, end='')
            print(f'{score_measures[i]}: {score}')

In [ ]:
query_with_score('发生了3人死亡，1000万元财产损失的安全事故，依法对涉事人员如何处理？')

## 注意 （永远显式指定度量标准）
### 1. 余弦相似度（cosine）：得分范围：-1 到 1
#### • 1：完全相同方向（最相似）
#### • 0：正交（不相关）
#### • -1：完全相反方向（最不相似）
### 2. 欧几里得距离（l2）得分范围：0 到 ∞（实际有上限）
#### • 0：完全相同（距离为0）
#### • 值越小越相似
#### • 值越大越不相似
### 3. 内积（ip）得分范围：-∞ 到 ∞
#### • 正值越大越相似
#### • 负值表示方向相反
#### • 向量归一化后等同于余弦相似度
